# Задачи 6.2, 6.8, 6.11, 6.12

Тема: первая квадратичная форма, длины, углы, площади и ортогональные траектории.

В этом каркасе есть два режима:

1. **2D-плоскость параметров** $(u,v)$ — удобно для областей, семейств линий и ОДУ.
2. **3D-визуализация поверхности** $r(u,v)$ — если поверхность задана явно.

In [ ]:
# Базовые библиотеки для аналитики и визуализации
import numpy as np
import matplotlib.pyplot as plt
from math import sin, cos, tan, sqrt, pi

try:
    import sympy as sp
except ImportError:
    sp = None

EPS = 1e-9
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True

In [ ]:
def setup_2d(xlim=(-5, 5), ylim=(-5, 5), title=None, xlabel='x', ylabel='y'):
    """Создает 2D-плоскость с осями координат и равным масштабом."""
    fig, ax = plt.subplots()
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    return fig, ax


def plot_points_2d(ax, points, labels=None):
    """Рисует точки на 2D-графике."""
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], s=45)
        if label:
            ax.text(p[0], p[1], '  ' + label)


def plot_parametric_2d(ax, xy_func, t_range, n=800, label=None):
    """Рисует плоскую параметрическую кривую t -> (x(t), y(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    xy = np.asarray(xy_func(t), dtype=float)
    ax.plot(xy[0], xy[1], label=label)
    if label:
        ax.legend()
    return xy


def plot_implicit_2d(ax, F, xlim, ylim, n=500, level=0, label=None):
    """Рисует неявную кривую F(x,y)=level через contour."""
    xs = np.linspace(xlim[0], xlim[1], n)
    ys = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(xs, ys)
    Z = F(X, Y)
    cs = ax.contour(X, Y, Z, levels=[level])
    if label:
        cs.collections[0].set_label(label)
        ax.legend()
    return cs

In [ ]:
def set_axes_equal_3d(ax):
    """Делает масштабы по осям 3D одинаковыми."""
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])
    radius = 0.5 * max([x_range, y_range, z_range])
    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)
    ax.set_xlim3d([x_middle - radius, x_middle + radius])
    ax.set_ylim3d([y_middle - radius, y_middle + radius])
    ax.set_zlim3d([z_middle - radius, z_middle + radius])


def setup_3d(xlim=(-5, 5), ylim=(-5, 5), zlim=(-5, 5), title=None):
    """Создает 3D-систему координат."""
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    if title:
        ax.set_title(title)
    # оси координат
    ax.plot([xlim[0], xlim[1]], [0, 0], [0, 0], linewidth=1)
    ax.plot([0, 0], [ylim[0], ylim[1]], [0, 0], linewidth=1)
    ax.plot([0, 0], [0, 0], [zlim[0], zlim[1]], linewidth=1)
    return fig, ax


def plot_points_3d(ax, points, labels=None):
    labels = labels or [None] * len(points)
    for p, label in zip(points, labels):
        p = np.asarray(p, dtype=float)
        ax.scatter(p[0], p[1], p[2], s=45)
        if label:
            ax.text(p[0], p[1], p[2], '  ' + label)


def plot_parametric_3d(ax, r_func, t_range, n=800, label=None):
    """Рисует пространственную параметрическую кривую t -> (x(t), y(t), z(t))."""
    t = np.linspace(t_range[0], t_range[1], n)
    r = np.asarray(r_func(t), dtype=float)
    ax.plot(r[0], r[1], r[2], label=label)
    if label:
        ax.legend()
    return r


def plot_surface_3d(ax, r_func, u_range, v_range, nu=80, nv=80, alpha=0.45, label=None):
    """Рисует параметрическую поверхность r(u,v). r_func должен возвращать массивы X,Y,Z."""
    u = np.linspace(u_range[0], u_range[1], nu)
    v = np.linspace(v_range[0], v_range[1], nv)
    U, V = np.meshgrid(u, v)
    X, Y, Z = r_func(U, V)
    surf = ax.plot_surface(X, Y, Z, alpha=alpha, linewidth=0, antialiased=True)
    if label:
        surf.set_label(label)
    set_axes_equal_3d(ax)
    return X, Y, Z

In [ ]:
def partial_u(r, u, v, h=1e-5):
    return (np.asarray(r(u + h, v)) - np.asarray(r(u - h, v))) / (2*h)


def partial_v(r, u, v, h=1e-5):
    return (np.asarray(r(u, v + h)) - np.asarray(r(u, v - h))) / (2*h)


def second_uu(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v)) - 2*np.asarray(r(u, v)) + np.asarray(r(u - h, v))) / (h*h)


def second_uv(r, u, v, h=1e-4):
    return (np.asarray(r(u + h, v + h)) - np.asarray(r(u + h, v - h)) - np.asarray(r(u - h, v + h)) + np.asarray(r(u - h, v - h))) / (4*h*h)


def second_vv(r, u, v, h=1e-4):
    return (np.asarray(r(u, v + h)) - 2*np.asarray(r(u, v)) + np.asarray(r(u, v - h))) / (h*h)


def surface_normal(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    return normalize(np.cross(ru, rv))


def first_fundamental_form(r, u, v):
    ru = partial_u(r, u, v)
    rv = partial_v(r, u, v)
    E = float(np.dot(ru, ru))
    F = float(np.dot(ru, rv))
    G = float(np.dot(rv, rv))
    return np.array([[E, F], [F, G]])


def second_fundamental_form(r, u, v):
    m = surface_normal(r, u, v)
    L = float(np.dot(second_uu(r, u, v), m))
    M = float(np.dot(second_uv(r, u, v), m))
    N = float(np.dot(second_vv(r, u, v), m))
    return np.array([[L, M], [M, N]])


def tangent_plane_patch(r, u0, v0, su=1.0, sv=1.0, n=12):
    """Патч касательной плоскости через r(u0,v0)."""
    p = np.asarray(r(u0, v0), dtype=float)
    ru = partial_u(r, u0, v0)
    rv = partial_v(r, u0, v0)
    a = np.linspace(-su, su, n)
    b = np.linspace(-sv, sv, n)
    A, B = np.meshgrid(a, b)
    X = p[0] + A*ru[0] + B*rv[0]
    Y = p[1] + A*ru[1] + B*rv[1]
    Z = p[2] + A*ru[2] + B*rv[2]
    return X, Y, Z


def plot_tangent_plane_and_normal(ax, r, u0, v0, plane_scale=0.5, normal_scale=1.0):
    p = np.asarray(r(u0, v0), dtype=float)
    X, Y, Z = tangent_plane_patch(r, u0, v0, plane_scale, plane_scale)
    ax.plot_surface(X, Y, Z, alpha=0.30, linewidth=0)
    m = surface_normal(r, u0, v0)
    ax.quiver(p[0], p[1], p[2], normal_scale*m[0], normal_scale*m[1], normal_scale*m[2], arrow_length_ratio=0.15)
    plot_points_3d(ax, [p], ['M'])
    set_axes_equal_3d(ax)
    return p, m

## Задача 6.2 — первая квадратичная форма поверхностей, построенных по кривой

In [ ]:
# Универсальная функция уже есть: first_fundamental_form(r, u, v).
# Ниже пример для проверки на конкретной кривой gamma(l).

# Пример: gamma(l) — окружность радиуса 1, параметр l натуральный.
def gamma(l):
    return np.array([np.cos(l), np.sin(l), 0.0], dtype=float)

# Постоянный вектор для цилиндрической поверхности.
a_vec = np.array([0.0, 0.0, 1.0])

def surface_cylindrical(l, lam):
    return gamma(l) + lam * a_vec

# Линейчатая поверхность: e(l) — единичный вектор.
def e_vec(l):
    return normalize(np.array([np.cos(l), np.sin(l), 1.0], dtype=float))

def surface_ruled(l, lam):
    return gamma(l) + lam * e_vec(l)

for name, surf in [('цилиндрическая', surface_cylindrical), ('линейчатая', surface_ruled)]:
    G = first_fundamental_form(surf, 0.7, 0.4)
    print(name, '\nG=\n', G)

fig, ax = setup_3d((-2, 2), (-2, 2), (-2, 2), title='6.2 пример: цилиндрическая поверхность')
plot_surface_3d(ax, surface_cylindrical, (0, 2*np.pi), (-1.5, 1.5), alpha=0.45)
plt.show()

### Место для аналитического вывода 6.2

Используй формулы Френе для натурально параметризованной кривой:

$$\gamma'=T,\quad T'=kn,\quad n'=-kT+\kappa b,\quad b'=-\kappa n.$$

Далее для каждой поверхности вычисляй $r_l$, $r_\lambda$ или $r_\varphi$ и матрицу

$$G=\begin{pmatrix}(r_1,r_1)&(r_1,r_2)\\(r_2,r_1)&(r_2,r_2)\end{pmatrix}.$$

## Задача 6.8 — ортогональные траектории на сфере

In [ ]:
R = 2.0

# В методичке у сферы, вероятно, опечатка: две первые координаты напечатаны одинаково.
# Используем стандартную параметризацию сферы широтой u и долготой v.
def sphere_uv(u, v, R=R):
    return (R*np.cos(u)*np.cos(v), R*np.cos(u)*np.sin(v), R*np.sin(u))

# Семейство исходных линий: u+v=C -> v=C-u.
# Для стандартной сферической метрики E=R^2, F=0, G=R^2 cos^2 u.
# Ортогональные траектории имеют dv/du = sec^2(u), то есть v = tan(u)+C.

def family_line(C, u):
    return C - u


def orthogonal_line(C, u):
    return np.tan(u) + C

fig, ax = setup_2d((-1.2, 1.2), (-4, 4), title='6.8 Семейства в плоскости параметров $(u,v)$', xlabel='u', ylabel='v')
u = np.linspace(-1.1, 1.1, 500)
for C in np.linspace(-2.0, 2.0, 5):
    ax.plot(u, family_line(C, u), label='u+v=C' if C == -2.0 else None)
for C in np.linspace(-2.0, 2.0, 5):
    ax.plot(u, orthogonal_line(C, u), linestyle='--', label='orthogonal' if C == -2.0 else None)
ax.legend()
plt.show()

# 3D-проверка на сфере
fig, ax = setup_3d((-2.2, 2.2), (-2.2, 2.2), (-2.2, 2.2), title='6.8 Линии на сфере')
plot_surface_3d(ax, sphere_uv, (-1.2, 1.2), (-np.pi, np.pi), alpha=0.18)
for C in np.linspace(-2.0, 2.0, 5):
    def r_line(t, C=C):
        v = family_line(C, t)
        return np.vstack(sphere_uv(t, v))
    plot_parametric_3d(ax, r_line, (-1.1, 1.1), n=300)
for C in np.linspace(-2.0, 2.0, 5):
    def r_orth(t, C=C):
        v = orthogonal_line(C, t)
        return np.vstack(sphere_uv(t, v))
    plot_parametric_3d(ax, r_orth, (-1.1, 1.1), n=300)
set_axes_equal_3d(ax)
plt.show()

## Задача 6.11 — треугольники по первой квадратичной форме

In [ ]:
# Метрика: ds^2 = du^2 + (u^2+a^2) dv^2.
a = 1.0

def metric_611(u, v, a=a):
    E = 1.0
    F = 0.0
    G = u**2 + a**2
    return E, F, G

# Визуализация областей в плоскости параметров.
v = np.linspace(0, 1, 400)
u_left = -0.5*a*v**2
u_right = 0.5*a*v**2

fig, ax = setup_2d((-0.8, 0.8), (-0.1, 1.2), title='6.11а-б область: u=±(a/2)v^2, v=1', xlabel='u', ylabel='v')
ax.plot(u_left, v, label=r'$u=-\frac{a}{2}v^2$')
ax.plot(u_right, v, label=r'$u=\frac{a}{2}v^2$')
ax.plot([u_left[-1], u_right[-1]], [1, 1], label='$v=1$')
ax.fill_betweenx(v, u_left, u_right, alpha=0.2)
ax.legend()
plt.show()

v = np.linspace(0, 1, 400)
u_left = -a*v
u_right = a*v
fig, ax = setup_2d((-1.2, 1.2), (-0.1, 1.2), title='6.11в область: u=±av, v=1', xlabel='u', ylabel='v')
ax.plot(u_left, v, label='$u=-av$')
ax.plot(u_right, v, label='$u=av$')
ax.plot([u_left[-1], u_right[-1]], [1, 1], label='$v=1$')
ax.fill_betweenx(v, u_left, u_right, alpha=0.2)
ax.legend()
plt.show()

### Место для вычислений 6.11

- Длина кривой $(u(t),v(t))$:

$$L=\int\sqrt{E\dot u^2+2F\dot u\dot v+G\dot v^2}\,dt.$$

- Угол между касательными векторами $\xi=(\xi^1,\xi^2)$ и $\eta=(\eta^1,\eta^2)$:

$$\cos\alpha=\frac{E\xi^1\eta^1+F(\xi^1\eta^2+\xi^2\eta^1)+G\xi^2\eta^2}{\sqrt{G(\xi,\xi)}\sqrt{G(\eta,\eta)}}.$$

- Площадь области $D$:

$$S=\iint_D \sqrt{EG-F^2}\,du\,dv.$$

## Задача 6.12 — поверхность $r(u,v)=(u\sin v,u\cos v,v)$

In [ ]:
v0 = 1.4

def surface_612(u, v):
    return (u*np.sin(v), u*np.cos(v), v)

# Область: 0 <= u <= sinh(v), 0 <= v <= v0.
def boundary_u0(t):
    return np.vstack(surface_612(np.zeros_like(t), t))

def boundary_v0(t):
    return np.vstack(surface_612(t, v0*np.ones_like(t)))

def boundary_sinh(t):
    return np.vstack(surface_612(np.sinh(t), t))

fig, ax = setup_2d((-0.1, np.sinh(v0)+0.2), (-0.1, v0+0.2), title='6.12 область в параметрах', xlabel='u', ylabel='v')
v = np.linspace(0, v0, 400)
ax.plot(np.zeros_like(v), v, label='$u=0$')
ax.plot(np.sinh(v), v, label='$u=\sinh v$')
ax.plot([0, np.sinh(v0)], [v0, v0], label='$v=v_0$')
ax.fill_betweenx(v, 0, np.sinh(v), alpha=0.2)
ax.legend()
plt.show()

fig, ax = setup_3d((-2, 2), (-2, 2), (-0.1, v0+0.3), title='6.12 поверхность и криволинейный треугольник')
plot_surface_3d(ax, surface_612, (0, np.sinh(v0)), (0, v0), alpha=0.25)
plot_parametric_3d(ax, boundary_u0, (0, v0), label='$u=0$')
plot_parametric_3d(ax, boundary_v0, (0, np.sinh(v0)), label='$v=v_0$')
plot_parametric_3d(ax, boundary_sinh, (0, v0), label='$u=\sinh v$')
plt.show()

# Проверка первой квадратичной формы в произвольной точке
print('G(u=0.5, v=0.7)=\n', first_fundamental_form(surface_612, 0.5, 0.7))